[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/01_what_is_machine_learning.ipynb)

# Bài 01 — Machine Learning là gì?

## 1. Giới thiệu

Machine Learning (học máy) là một lĩnh vực của trí tuệ nhân tạo (AI). Mục tiêu của Machine Learning là để máy tính **học được quy luật từ dữ liệu** — thay vì được lập trình từng quy tắc — để dự đoán hoặc ra quyết định trên dữ liệu mới.

<img src="images/Deep_Learning_Icons_R5.png" style="width:60%;height:60%;">

```text
Trí tuệ nhân tạo (AI)  ⊃  Machine Learning  ⊃  Deep Learning
   máy làm việc "thông minh"   học từ dữ liệu        mạng neural nhiều tầng
```

## 2. Machine Learning khác lập trình truyền thống thế nào?

Trong lập trình truyền thống, thuật toán được lập trình **rõ ràng** cho từng vấn đề. Với Machine Learning, ta dùng dữ liệu có sẵn để **phân tích — thống kê và tìm ra quy luật**.

<img src="images/ml-ex.png" style="width:60%;height:60%;">

Xét bài toán nhận diện hoạt động từ tốc độ di chuyển, theo cách truyền thống:

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
WALKING, RUNNING, BIKING = "walking", "running", "biking"


def classify_activity(speed):
    if speed < 4:
        return WALKING
    elif speed < 12:
        return RUNNING
    else:
        return BIKING


print([classify_activity(s) for s in [3, 8, 20]])

Với một số ít hoạt động như đi bộ, chạy bộ, đi xe đạp, ta có thể dùng vài lệnh `if – else`. Nhưng thêm hoạt động **chơi golf** thì sao? Tốc độ khi chơi golf trùng với đi bộ; cần thêm nhịp tim, gia tốc, kiểu chuyển động... Rất khó để viết tay các quy tắc chính xác — nhưng Machine Learning có thể **tự học** chúng từ dữ liệu.

<img src="images/ml-ex2.png" style="width:45%;height:45%;"> <img src="images/ml-ex3.png" style="width:45%;height:45%;">

- **Lập trình truyền thống:** Rules + Data → Answers
- **Machine Learning (học có giám sát):** Data + Answers → Rules

Ta thu thập thật nhiều dữ liệu về các hoạt động và **gán nhãn** cho từng mẫu (dữ liệu này là đi bộ, dữ liệu kia là chơi golf):

<img src="images/ml-ex4.png" style="width:60%;height:60%;">

Từ dữ liệu có nhãn, Machine Learning tìm ra quy luật mà không cần lập trình cụ thể; với dữ liệu mới, hệ thống đưa ra dự đoán:

<img src="images/ml-ex5.png" style="width:45%;height:45%;">

### Thử nghiệm *(bổ sung)*: để máy tự học quy tắc

Mô phỏng dữ liệu cảm biến cho 4 hoạt động (tốc độ, nhịp tim, độ biến thiên gia tốc) rồi để một Decision Tree tự học quy tắc:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

rng = np.random.default_rng(0)
profiles = {  # (tốc độ km/h, nhịp tim, biến thiên gia tốc) — trung bình & độ lệch
    "walking": ([4.5, 95, 1.0], [1.0, 8, 0.3]),
    "running": ([10, 150, 2.5], [2.0, 12, 0.5]),
    "biking": ([20, 130, 1.2], [4.0, 12, 0.4]),
    "golfing": ([3.5, 100, 3.5], [1.2, 10, 0.8]),   # chậm như đi bộ nhưng có cú swing mạnh
}
rows = []
for label, (mu, sd) in profiles.items():
    data = rng.normal(mu, sd, size=(300, 3))
    rows += [(*r, label) for r in data]
df = pd.DataFrame(rows, columns=["speed", "heart_rate", "accel_var", "activity"])

X_train, X_test, y_train, y_test = train_test_split(df[["speed", "heart_rate", "accel_var"]], df["activity"],
                                                    test_size=0.25, random_state=0, stratify=df["activity"])
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(f"Accuracy trên dữ liệu MỚI (test): {tree.score(X_test, y_test):.3f}\n")
print(export_text(tree, feature_names=["speed", "heart_rate", "accel_var"]))

In [ ]:
rule_based = np.array([classify_activity(s) for s in X_test["speed"]])
print(f"Quy tắc viết tay (chỉ dùng tốc độ): accuracy = {np.mean(rule_based == y_test):.3f} — không thể nhận ra golf")

Model tự tìm ra các ngưỡng (vd: `accel_var` lớn ⇒ golf) — những quy tắc con người khó nghĩ ra. Đó là sức mạnh của ML: **khi quy tắc quá phức tạp để viết tay nhưng có thể học từ dữ liệu**.

## 3. Một số định nghĩa về Machine Learning

**Arthur Samuel (1959):**
> *"The field of study that gives computers the ability to learn without being explicitly programmed."*

Học máy là lĩnh vực nghiên cứu cung cấp cho máy tính khả năng học hỏi mà không cần phải lập trình rõ ràng.

**Tom Mitchell (1997)** — định nghĩa hiện đại và "kỹ thuật" hơn:
> *"A computer program is said to learn from experience E with respect to some class of tasks T and performance measure P, if its performance at tasks in T, as measured by P, improves with experience E."*

Một chương trình học từ kinh nghiệm **E** để làm nhiệm vụ **T**, được đo bởi thước đo **P**, nếu hiệu suất tại **T** (đo bằng **P**) cải thiện theo kinh nghiệm **E**.

| Bài toán | T (nhiệm vụ) | P (thước đo) | E (kinh nghiệm) |
|---|---|---|---|
| Chơi cờ | chơi cờ | xác suất thắng ván tiếp theo | các ván đã chơi |
| Lọc spam | phân loại email | % email phân loại đúng | email đã được gán nhãn spam/không |
| Dự đoán giá nhà | ước lượng giá | sai số tuyệt đối trung bình | giao dịch nhà trong quá khứ |
| Ví dụ trên | nhận diện hoạt động | accuracy | dữ liệu cảm biến có nhãn |

> 💡 Định nghĩa của Mitchell nhắc ta 3 điều phải xác định **trước** khi làm bất kỳ dự án ML nào: nhiệm vụ cụ thể, thước đo thành công, và dữ liệu để học.

## 4. Một số ứng dụng của Machine Learning

Nhận diện khuôn mặt (mở khoá điện thoại), gợi ý phim/nhạc (Netflix, Spotify), gợi ý sản phẩm (Tiki, Shopee, Lazada), lọc spam, phát hiện gian lận thẻ, chấm điểm tín dụng, dịch máy, trợ lý ảo và mô hình ngôn ngữ lớn, xe tự hành, chẩn đoán hình ảnh y tế, dự báo nhu cầu hàng hoá...

## 5. Khi nào KHÔNG nên dùng Machine Learning? *(bổ sung)*

- Quy tắc **đơn giản, rõ ràng, ổn định** → viết code thường (rẻ hơn, dễ kiểm chứng).
- **Không có dữ liệu** (hoặc không có nhãn) và không thể thu thập.
- Cần **giải thích tuyệt đối / đảm bảo 100%** (vd: tính thuế theo luật) — model ML luôn có sai số.
- Chi phí của sai sót rất lớn mà không có con người kiểm tra lại.
- Dữ liệu tương lai khác hẳn quá khứ (quy luật thay đổi liên tục).

> *"Rule #1: Don't be afraid to launch a product without machine learning."* — Google, *Rules of Machine Learning*

## 6. Quy trình một dự án Machine Learning *(bổ sung)*

```text
Xác định bài toán (T, P, E)
   ↓
Thu thập & hiểu dữ liệu (EDA — chương 03)
   ↓
Tiền xử lý & feature engineering (chương 04, 17)
   ↓
Chọn model, train, đánh giá trên validation (chương 06–16)
   ↓
Kiểm tra cuối trên test — một lần (chương 05, 15)
   ↓
Triển khai, giám sát, cập nhật (chương 39–41)
```

## 🧠 Kiểm tra nhanh

1. Xác định T, P, E cho bài toán "dự đoán khách hàng rời bỏ dịch vụ".
2. Vì sao quy tắc `if speed < 4` không nhận diện được hoạt động chơi golf?
3. Cho một ví dụ bài toán **không nên** dùng ML.

## Tài liệu tham khảo

[1] [Learn TensorFlow 1: The "Hello World" of machine learning](https://codelabs.developers.google.com/codelabs/tensorflow-lab1-helloworld/) · [2] Tom Mitchell — *Machine Learning* (1997) · [3] Google — [Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml)

---
<sub>Bài gốc: *Giới thiệu về Machine Learning* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 2 (MIT License). Đã bổ sung: thí nghiệm để máy tự học quy tắc, bảng T/P/E, khi nào không nên dùng ML, quy trình dự án.</sub>